In [ ]:
"""
============================================================
  TDNN (Time Delay Neural Network) — Javanese ASR
  Perbandingan dengan GMM pipeline
============================================================
  Label    : Word-level per frame (uniform segmentation)
             Vocab = kata unik dari seluruh dataset
  Input    : .wav dari SAMPLED_ROOT + dataSampledSmall.csv
  Output   : WER + Akurasi (identik format GMM)
============================================================
"""

import os, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────
#  KONFIGURASI
# ─────────────────────────────────────────────────────────────

SAMPLED_ROOT = r"D:\doc\2026\FINAL_CODE\data\sampled_small"
CSV_PATH = (
    r"D:/doc/2026/FINAL_CODE/src/codeStat/preModelling/"
    r"samplingStratified/out_samplingStratified/dataSampledSmall.csv"
)
DIR_SAVE = (
    r"D:/doc/2026/FINAL_CODE/src/codeStat/modelling/"
    r"voiceModelling/srcVoiceModelling/TDNN/out/test_2"
)
os.makedirs(DIR_SAVE, exist_ok=True)

OUT_WER_CSV     = os.path.join(DIR_SAVE, "HasilWER_TDNN.csv")
OUT_DEC_CSV     = os.path.join(DIR_SAVE, "HasilDekode_TDNN.csv")
OUT_ACC_CSV     = os.path.join(DIR_SAVE, "HasilAkurasi_TDNN.csv")
OUT_HISTORY_CSV = os.path.join(DIR_SAVE, "HasilTraining_TDNN.csv")
MODEL_PATH      = os.path.join(DIR_SAVE, "tdnn_best.npy")

COL_PATH_DATA = "Path_Data"
COL_SENTENCES = "sentences"

# ── MFCC (identik GMM) ──
SR=16000; N_MFCC=13; N_FFT=512; HOP_LENGTH=160
WIN_LENGTH=400; N_MELS=26; PRE_EMPH=0.97
FMIN=0.0; FMAX=4000.0; DELTA_WIDTH=2
INPUT_DIM = N_MFCC * 3   # 39

# ── Arsitektur ──
TDNN_CONTEXTS = [[-2,2],[-2,0],[-3,3],[0,0]]
HIDDEN_DIMS   = [512, 512, 512, 512]
DROPOUT_RATE  = 0.1
LEARNING_RATE = 0.001

# ── Training ──
BATCH_SIZE  = 512    # frame per batch
MAX_EPOCHS  = 30
EARLY_STOP  = 5
GRAD_CLIP   = 5.0

# ── Split ──
TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10
RANDOM_SEED = 42


# ─────────────────────────────────────────────────────────────
#  EKSTRAKSI MFCC — VECTORIZED (identik GMM, tanpa Python loop)
# ─────────────────────────────────────────────────────────────

try:
    import librosa
    LIBROSA_OK = True
except ImportError:
    LIBROSA_OK = False
try:
    from scipy.io import wavfile
    SCIPY_OK = True
except ImportError:
    SCIPY_OK = False


def extract_features(wav_path: str) -> np.ndarray:
    """Ekstrak MFCC 39-dim (13 + delta + delta2) dari .wav."""
    if not os.path.isfile(wav_path):
        raise FileNotFoundError(wav_path)
    return _librosa(wav_path) if LIBROSA_OK else _manual(wav_path)


def _librosa(wav_path):
    y, _  = librosa.load(wav_path, sr=SR, mono=True)
    if len(y) == 0:
        raise ValueError("Audio kosong")
    y     = np.append(y[0], y[1:] - PRE_EMPH * y[:-1])
    mfcc  = librosa.feature.mfcc(
        y=y, sr=SR, n_mfcc=N_MFCC, n_fft=N_FFT,
        hop_length=HOP_LENGTH, win_length=WIN_LENGTH,
        n_mels=N_MELS, fmin=FMIN, fmax=FMAX, window="hamming"
    )  # (13, T)
    d1 = librosa.feature.delta(mfcc, width=DELTA_WIDTH*2+1)
    d2 = librosa.feature.delta(mfcc, order=2, width=DELTA_WIDTH*2+1)
    return np.vstack([mfcc, d1, d2]).T.astype(np.float32)  # (T,39)


def _delta_vec(feat: np.ndarray, width: int = 2) -> np.ndarray:
    """Delta vectorized — tanpa Python loop antar frame."""
    T     = feat.shape[0]
    denom = 2.0 * sum(n*n for n in range(1, width+1))
    delta = np.zeros_like(feat)
    for n in range(1, width+1):
        # geser maju dan mundur, clamp di tepi
        fwd = feat[np.clip(np.arange(T)+n, 0, T-1)]
        bwd = feat[np.clip(np.arange(T)-n, 0, T-1)]
        delta += n * (fwd - bwd)
    return delta / denom


def _manual(wav_path):
    if not SCIPY_OK:
        raise ImportError("Butuh librosa atau scipy")
    sr, audio = wavfile.read(wav_path)
    if audio.dtype == np.int16:
        audio = audio / 32768.0
    elif audio.dtype == np.int32:
        audio = audio / 2147483648.0
    audio = audio.astype(np.float32)
    if sr != SR:
        n = int(len(audio)*SR/sr)
        audio = np.interp(np.linspace(0,len(audio)-1,n), np.arange(len(audio)), audio)
    y     = np.append(audio[0], audio[1:] - PRE_EMPH*audio[:-1])
    # Framing vectorized
    T_fr  = 1 + (len(y) - WIN_LENGTH) // HOP_LENGTH
    idx   = (np.arange(WIN_LENGTH)[None,:] +
             np.arange(T_fr)[:,None]*HOP_LENGTH)
    frames = y[idx] * np.hamming(WIN_LENGTH)
    fft_m  = np.abs(np.fft.rfft(frames, n=N_FFT))
    # Mel filterbank
    def hz2mel(f): return 1127*np.log1p(f/700)
    def mel2hz(m): return 700*(np.exp(m/1127)-1)
    pts  = mel2hz(np.linspace(hz2mel(FMIN), hz2mel(FMAX), N_MELS+2))
    bins = np.floor((N_FFT+1)*pts/SR).astype(int)
    fb   = np.zeros((N_MELS, N_FFT//2+1))
    for m in range(1, N_MELS+1):
        for k in range(bins[m-1], bins[m]):
            fb[m-1,k] = (k-bins[m-1])/(bins[m]-bins[m-1]+1e-10)
        for k in range(bins[m], bins[m+1]):
            fb[m-1,k] = (bins[m+1]-k)/(bins[m+1]-bins[m]+1e-10)
    log_mel = np.log10(np.maximum(fft_m**2 @ fb.T, 1e-10))
    # DCT
    N_     = N_MELS
    ni     = np.arange(N_MFCC)[:,None]
    ki     = np.arange(N_)[None,:] + 0.5
    dct_m  = np.cos(np.pi * ni * ki / N_) * np.sqrt(2/N_)
    mfcc   = log_mel @ dct_m.T
    d1     = _delta_vec(mfcc, DELTA_WIDTH)
    d2     = _delta_vec(d1,   DELTA_WIDTH)
    return np.concatenate([mfcc, d1, d2], axis=1).astype(np.float32)


# ─────────────────────────────────────────────────────────────
#  UNIFORM WORD SEGMENTATION
# ─────────────────────────────────────────────────────────────

def uniform_word_labels(words: list, T: int, word2id: dict) -> np.ndarray:
    """
    Bagi T frame ke setiap kata secara proporsional.

    Contoh: words=["aku","arep","pasar"], T=10
      frame 0-2 → "aku", frame 3-5 → "arep", frame 6-9 → "pasar"

    Metodologi (untuk skripsi):
      Karena tidak ada forced alignment, durasi setiap kata
      diasumsikan proporsional terhadap jumlah kata (uniform).
    """
    N      = len(words)
    labels = np.zeros(T, dtype=np.int32)
    if N == 0:
        return labels
    base = T // N
    rem  = T % N
    start = 0
    for i, w in enumerate(words):
        end = start + base + (1 if i < rem else 0)
        labels[start:end] = word2id.get(w, word2id["<unk>"])
        start = end
    return labels


# ─────────────────────────────────────────────────────────────
#  DATA LOADER
# ─────────────────────────────────────────────────────────────

def find_wav(audio_id: str) -> str:
    for root, _, files in os.walk(SAMPLED_ROOT):
        for f in files:
            if os.path.splitext(f)[0] == audio_id and f.lower().endswith(".wav"):
                return os.path.join(root, f)
    return ""


def load_data(csv_path: str) -> tuple:
    """
    Baca CSV → ekstrak MFCC per frame → label kata per frame.

    Output dim = jumlah kata unik (vocab), bukan jumlah kalimat.
    """
    print(f"  Membaca {csv_path} ...")
    df = pd.read_csv(csv_path)
    print(f"  Shape CSV : {df.shape}")
    print(f"  Kolom     : {list(df.columns)}")

    for col in [COL_PATH_DATA, COL_SENTENCES]:
        if col not in df.columns:
            raise ValueError(f"Kolom '{col}' tidak ada. Ada: {list(df.columns)}")

    # ── Bangun vocab dari KATA (bukan kalimat) ────────────────
    all_words = []
    for sent in df[COL_SENTENCES].astype(str).str.lower().str.strip():
        all_words.extend(sent.split())

    vocab   = ["<unk>"] + sorted(set(all_words))
    word2id = {w: i for i, w in enumerate(vocab)}
    id2word = {i: w for w, i in word2id.items()}
    print(f"  Vocab kata unik  : {len(vocab)-1}  (+1 <unk>  → total {len(vocab)})")

    # ── Ekstrak MFCC + label per frame ───────────────────────
    sequences, labels, word_refs = {}, {}, {}
    skipped = 0

    print(f"  Mengekstrak MFCC dari {len(df)} file .wav ...")
    for idx, row in df.iterrows():
        audio_id = os.path.splitext(os.path.basename(str(row[COL_PATH_DATA])))[0]
        kalimat  = str(row[COL_SENTENCES]).lower().strip()
        words    = kalimat.split()

        wav_path = str(row[COL_PATH_DATA])
        if not os.path.isfile(wav_path):
            wav_path = find_wav(audio_id)
        if not wav_path:
            skipped += 1; continue

        try:
            feats = extract_features(wav_path)   # (T, 39)
            T     = feats.shape[0]
            if T == 0 or not words:
                skipped += 1; continue

            sequences[audio_id] = feats
            labels[audio_id]    = uniform_word_labels(words, T, word2id)
            word_refs[audio_id] = words

        except Exception as e:
            print(f"  ⚠ {audio_id}: {e}")
            skipped += 1; continue

        if (idx+1) % 200 == 0 or (idx+1) == len(df):
            print(f"    [{idx+1}/{len(df)}] diproses, {skipped} dilewati")

    total_frames = sum(v.shape[0] for v in sequences.values())
    actual_dim   = next(iter(sequences.values())).shape[1]

    print(f"\n  Audio berhasil : {len(sequences)}")
    print(f"  Audio dilewati : {skipped}")
    print(f"  Total frame    : {total_frames:,}")
    print(f"  Input dim      : {actual_dim}")
    print(f"  Output dim     : {len(vocab)}  ← vocab kata, bukan kalimat")

    return sequences, labels, word2id, id2word, word_refs, actual_dim


# ─────────────────────────────────────────────────────────────
#  AKTIVASI & LOSS
# ─────────────────────────────────────────────────────────────

def relu(x):      return np.maximum(0.0, x)
def relu_grad(x): return (x > 0).astype(np.float32)

def softmax(x):
    x = x - x.max(axis=-1, keepdims=True)
    e = np.exp(x)
    return e / (e.sum(axis=-1, keepdims=True) + 1e-9)

def ce_loss(p, y):
    return -np.log(p[np.arange(len(y)), y] + 1e-9).mean()

def accuracy(p, y):
    return (np.argmax(p, axis=1) == y).mean()


# ─────────────────────────────────────────────────────────────
#  TDNN LAYER
# ─────────────────────────────────────────────────────────────

class TDNNLayer:
    def __init__(self, in_dim, out_dim, context, lr):
        self.context = context
        self.ctx_len = context[1] - context[0] + 1
        self.lr      = lr
        fan_in       = in_dim * self.ctx_len
        self.W = (np.random.randn(fan_in, out_dim) *
                  np.sqrt(2.0/fan_in)).astype(np.float32)
        self.b = np.zeros(out_dim, dtype=np.float32)
        self.mW=np.zeros_like(self.W); self.vW=np.zeros_like(self.W)
        self.mb=np.zeros_like(self.b); self.vb=np.zeros_like(self.b)
        self.t=0; self._X=None

    def splice(self, X):
        """Index-clamp splicing — aman untuk T=1."""
        T   = X.shape[0]
        idx = np.arange(T)
        return np.concatenate(
            [X[np.clip(idx+off, 0, T-1)]
             for off in range(self.context[0], self.context[1]+1)],
            axis=1
        )

    def forward(self, Xsp):
        self._X = Xsp
        return Xsp @ self.W + self.b

    def backward(self, dout):
        dW = self._X.T @ dout / len(dout)
        db = dout.mean(0)
        dXsp = dout @ self.W.T                        # (B, D*ctx_len)
        # Gradient clip
        norm = np.sqrt((dW**2).sum() + (db**2).sum())
        if norm > GRAD_CLIP:
            dW *= GRAD_CLIP/norm; db *= GRAD_CLIP/norm
        self._adam(dW, db)
        # Un-splice: sum kontribusi semua offset → (B, D)
        B, D_tot = dXsp.shape
        D  = D_tot // self.ctx_len
        dX = np.zeros((B, D), dtype=np.float32)
        for k in range(self.ctx_len):
            dX += dXsp[:, k*D:(k+1)*D]
        return dX

    def _adam(self, dW, db, b1=0.9, b2=0.999, eps=1e-8):
        self.t += 1
        self.mW = b1*self.mW+(1-b1)*dW
        self.vW = b2*self.vW+(1-b2)*dW**2
        self.W -= self.lr*(self.mW/(1-b1**self.t))/(np.sqrt(self.vW/(1-b2**self.t))+eps)
        self.mb = b1*self.mb+(1-b1)*db
        self.vb = b2*self.vb+(1-b2)*db**2
        self.b -= self.lr*(self.mb/(1-b1**self.t))/(np.sqrt(self.vb/(1-b2**self.t))+eps)


# ─────────────────────────────────────────────────────────────
#  MODEL TDNN
# ─────────────────────────────────────────────────────────────

class TDNNModel:
    def __init__(self, input_dim, hidden_dims, contexts,
                 output_dim, dropout=0.1, lr=0.001):
        assert len(hidden_dims) == len(contexts)
        self.dropout  = dropout
        self.training = True
        self.layers   = []
        in_d = input_dim
        for h, ctx in zip(hidden_dims, contexts):
            self.layers.append(TDNNLayer(in_d, h, ctx, lr))
            in_d = h
        self.layers.append(TDNNLayer(in_d, output_dim, [0,0], lr))
        self._rc = []; self._dc = []   # relu cache, drop cache

    def forward(self, X):
        self._rc = []; self._dc = []
        h = X
        for layer in self.layers[:-1]:
            z = layer.forward(layer.splice(h))
            a = relu(z)
            if self.training and self.dropout > 0:
                mask = (np.random.rand(*a.shape) > self.dropout
                        ).astype(np.float32)
                a    = a * mask / (1.0 - self.dropout)
            else:
                mask = np.ones(a.shape, dtype=np.float32)
            self._rc.append(z); self._dc.append(mask)
            h = a
        out = self.layers[-1]
        return out.forward(out.splice(h))

    def loss_and_backward(self, X, y):
        logits = self.forward(X)
        probs  = softmax(logits)
        loss   = ce_loss(probs, y)
        acc    = accuracy(probs, y)
        dlog   = probs.copy()
        dlog[np.arange(len(y)), y] -= 1.0
        dh  = self.layers[-1].backward(dlog)
        sc  = 1.0/(1.0-self.dropout+1e-9) if self.dropout > 0 else 1.0
        for i in reversed(range(len(self.layers)-1)):
            dh = dh * self._dc[i] * sc
            dh = dh * relu_grad(self._rc[i])
            dh = self.layers[i].backward(dh)
        return loss, acc

    def predict_proba(self, X):
        self.training = False
        return softmax(self.forward(X))

    def save(self, path):
        np.save(path, {"w": [(l.W.copy(), l.b.copy()) for l in self.layers]},
                allow_pickle=True)

    def load(self, path):
        d = np.load(path, allow_pickle=True).item()
        for i, (W, b) in enumerate(d["w"]):
            self.layers[i].W = W; self.layers[i].b = b
        print(f"  Model dimuat: {path}")


# ─────────────────────────────────────────────────────────────
#  SPLIT
# ─────────────────────────────────────────────────────────────

def split_dataset(ids):
    rng  = np.random.default_rng(RANDOM_SEED)
    ids  = list(ids)
    perm = rng.permutation(len(ids))
    n    = len(ids)
    ntr  = int(n*TRAIN_RATIO); nva = int(n*VAL_RATIO)
    tr = [ids[i] for i in perm[:ntr]]
    va = [ids[i] for i in perm[ntr:ntr+nva]]
    te = [ids[i] for i in perm[ntr+nva:]]
    print(f"  Split → train:{len(tr)}  val:{len(va)}  test:{len(te)}")
    return tr, va, te


# ─────────────────────────────────────────────────────────────
#  TRAINING
# ─────────────────────────────────────────────────────────────

def train(model, train_ids, val_ids, sequences, labels):
    rng    = np.random.default_rng(RANDOM_SEED)
    hist   = []
    best   = float("inf")
    pat    = 0

    print(f"\n  Train: {len(train_ids)} audio | Val: {len(val_ids)} audio")
    print(f"  Epochs: {MAX_EPOCHS} | EarlyStop: {EARLY_STOP} | Batch: {BATCH_SIZE} frame")
    print(f"\n  {'Ep':>4}  {'TrLoss':>8}  {'TrAcc%':>8}  {'VaLoss':>8}  {'VaAcc%':>8}  {'t':>7}")
    print(f"  {'-'*55}")

    for epoch in range(1, MAX_EPOCHS+1):
        t0 = time.time()
        model.training = True

        # Kumpulkan semua frame train → shuffle global
        all_X = np.concatenate([sequences[a] for a in train_ids])
        all_y = np.concatenate([labels[a]    for a in train_ids])
        perm  = rng.permutation(len(all_X))
        all_X = all_X[perm]; all_y = all_y[perm]

        el = ea = nb = 0
        for s in range(0, len(all_X), BATCH_SIZE):
            Xb = all_X[s:s+BATCH_SIZE]
            yb = all_y[s:s+BATCH_SIZE]
            if len(Xb) < 2: continue
            loss, acc = model.loss_and_backward(Xb, yb)
            el += loss; ea += acc; nb += 1

        el /= max(nb,1); ea /= max(nb,1)

        # Validasi
        model.training = False
        val_X = np.concatenate([sequences[a] for a in val_ids])
        val_y = np.concatenate([labels[a]    for a in val_ids])
        vp    = model.predict_proba(val_X)
        vl    = ce_loss(vp, val_y)
        va    = accuracy(vp, val_y)

        elapsed = time.time()-t0
        hist.append({
            "epoch":epoch, "train_loss":round(el,6),
            "train_acc":round(ea*100,4),
            "val_loss":round(vl,6), "val_acc":round(va*100,4)
        })
        print(f"  {epoch:4d}  {el:8.4f}  {ea*100:8.4f}  "
              f"{vl:8.4f}  {va*100:8.4f}  {elapsed:6.1f}s")

        if vl < best - 1e-5:
            best = vl; pat = 0; model.save(MODEL_PATH)
        else:
            pat += 1
            if pat >= EARLY_STOP:
                print(f"\n  ✓ Early stop epoch {epoch} (best={best:.4f})")
                break

    return hist


# ─────────────────────────────────────────────────────────────
#  DEKODING (CTC-style collapse)
# ─────────────────────────────────────────────────────────────

def decode_to_words(probs, id2word):
    """Frame predictions → collapsed word sequence."""
    preds = np.argmax(probs, axis=1)
    words, prev = [], None
    for pid in preds:
        if pid != prev:
            w = id2word.get(int(pid), "<unk>")
            if w != "<unk>":
                words.append(w)
            prev = pid
    return words


# ─────────────────────────────────────────────────────────────
#  WER (Levenshtein)
# ─────────────────────────────────────────────────────────────

def levenshtein(ref, hyp):
    R, H = len(ref), len(hyp)
    dp   = np.zeros((R+1,H+1), dtype=int)
    dp[:,0] = np.arange(R+1); dp[0,:] = np.arange(H+1)
    for i in range(1,R+1):
        for j in range(1,H+1):
            c = 0 if ref[i-1]==hyp[j-1] else 1
            dp[i,j] = min(dp[i-1,j]+1, dp[i,j-1]+1, dp[i-1,j-1]+c)
    i,j = R,H; S=D=I=0
    while i>0 or j>0:
        if i>0 and j>0:
            c = 0 if ref[i-1]==hyp[j-1] else 1
            if dp[i,j]==dp[i-1,j-1]+c:
                if c: S+=1
                i-=1; j-=1
            elif dp[i,j]==dp[i-1,j]+1: D+=1; i-=1
            else: I+=1; j-=1
        elif i>0: D+=1; i-=1
        else: I+=1; j-=1
    return min((S+D+I)/max(R,1),1.0), S, D, I, max(R,1)


# ─────────────────────────────────────────────────────────────
#  EVALUASI
# ─────────────────────────────────────────────────────────────

def evaluate(model, eval_ids, sequences, labels, id2word, word_refs):
    model.training = False
    print(f"\n  Mendekode {len(eval_ids)} audio ...")
    wer_rows=[]; acc_rows=[]
    tS=tD=tI=tN=tC=tF=0

    for aid in eval_ids:
        X = sequences[aid]; y = labels[aid]; T = len(X)
        p = model.predict_proba(X)
        pr = np.argmax(p, axis=1)
        c  = int((pr==y).sum()); tC+=c; tF+=T
        hw = decode_to_words(p, id2word)
        rw = word_refs.get(aid, [])
        wv,S,D,I,N = levenshtein(rw, hw)
        tS+=S; tD+=D; tI+=I; tN+=N
        wer_rows.append({"audio_id":aid,"WER":round(wv*100,4),
                         "S":S,"D":D,"I":I,"N":N,
                         "referensi":" ".join(rw),
                         "hipotesis":" ".join(hw)})
        acc_rows.append({"audio_id":aid,
                         "akurasi_pct":round(c/T*100,4),
                         "benar":c,"total_frame":T})

    wc = (tS+tD+tI)/max(tN,1)*100
    ag = tC/max(tF,1)*100
    wl = [r["WER"] for r in wer_rows]
    al = [r["akurasi_pct"] for r in acc_rows]
    s  = {"n_audio":len(eval_ids),
          "wer_corpus":round(wc,4),
          "wer_mean":round(float(np.mean(wl)),4),
          "wer_min":round(float(np.min(wl)),4),
          "wer_max":round(float(np.max(wl)),4),
          "acc_global":round(ag,4),
          "acc_mean":round(float(np.mean(al)),4),
          "acc_min":round(float(np.min(al)),4),
          "acc_max":round(float(np.max(al)),4),
          "total_S":tS,"total_D":tD,"total_I":tI,"total_N":tN,
          "total_frame":tF,"total_benar":tC}

    print(f"\n── Statistik Evaluasi TDNN ────────────────────────────────")
    print(f"  Jumlah audio     : {s['n_audio']}")
    print(f"  ── WER ──────────────────────────────────────────────────")
    print(f"  WER Corpus       : {s['wer_corpus']:.2f}%")
    print(f"  WER Mean         : {s['wer_mean']:.2f}%")
    print(f"  WER Min          : {s['wer_min']:.2f}%")
    print(f"  WER Max          : {s['wer_max']:.2f}%")
    print(f"  Total S          : {s['total_S']}")
    print(f"  Total D          : {s['total_D']}")
    print(f"  Total I          : {s['total_I']}")
    print(f"  Total N (kata)   : {s['total_N']}")
    print(f"  ── Akurasi Frame ────────────────────────────────────────")
    print(f"  Akurasi Global   : {s['acc_global']:.2f}%")
    print(f"  Akurasi Mean     : {s['acc_mean']:.2f}%")
    print(f"  Akurasi Min      : {s['acc_min']:.2f}%")
    print(f"  Akurasi Max      : {s['acc_max']:.2f}%")
    print(f"  Total frame      : {s['total_frame']:,}")
    print(f"  Frame benar      : {s['total_benar']:,}")
    print(f"────────────────────────────────────────────────────────────")
    return wer_rows, acc_rows, s


# ─────────────────────────────────────────────────────────────
#  PERBANDINGAN GMM vs TDNN
# ─────────────────────────────────────────────────────────────

def print_comparison(s):
    g = {"wer_corpus":99.84,"wer_mean":99.82,"wer_min":50.00,"wer_max":100.00,
         "total_S":1730,"total_D":10394,"total_I":0,"total_N":12143}
    print(f"\n{'='*62}")
    print(f"  PERBANDINGAN GMM vs TDNN")
    print(f"{'='*62}")
    print(f"  {'Metrik':<30} {'GMM':>10} {'TDNN':>10}")
    print(f"  {'-'*52}")
    for lbl, gv, tv in [
        ("WER Corpus (%)",       f"{g['wer_corpus']:.2f}", f"{s['wer_corpus']:.2f}"),
        ("WER Mean (%)",         f"{g['wer_mean']:.2f}",   f"{s['wer_mean']:.2f}"),
        ("WER Min (%)",          f"{g['wer_min']:.2f}",    f"{s['wer_min']:.2f}"),
        ("WER Max (%)",          f"{g['wer_max']:.2f}",    f"{s['wer_max']:.2f}"),
        ("Total S (substitusi)", str(g['total_S']),        str(s['total_S'])),
        ("Total D (deletion)",   str(g['total_D']),        str(s['total_D'])),
        ("Total I (insertion)",  str(g['total_I']),        str(s['total_I'])),
        ("Total N (kata ref)",   str(g['total_N']),        str(s['total_N'])),
        ("Akurasi Frame (%)",    "N/A",                    f"{s['acc_global']:.2f}"),
    ]:
        print(f"  {lbl:<30} {gv:>10} {tv:>10}")
    print(f"{'='*62}")
    d = g['wer_corpus'] - s['wer_corpus']
    if d > 0:   print(f"  ✓ TDNN lebih baik {d:.2f}%")
    elif d < 0: print(f"  ✗ TDNN lebih buruk {abs(d):.2f}%")
    else:       print(f"  = WER sama")


# ─────────────────────────────────────────────────────────────
#  MAIN
# ─────────────────────────────────────────────────────────────

def main():
    np.random.seed(RANDOM_SEED)

    print("="*60)
    print("  TAHAP 1 — Memuat Data & Ekstraksi MFCC dari WAV")
    print("="*60)
    print(f"  Folder audio : {SAMPLED_ROOT}")
    print(f"  CSV          : {CSV_PATH}")
    print(f"  Output       : {DIR_SAVE}")
    print(f"  Label        : word-level, uniform segmentation")

    sequences, labels, word2id, id2word, word_refs, actual_dim = load_data(CSV_PATH)
    all_ids = list(sequences.keys())
    train_ids, val_ids, test_ids = split_dataset(all_ids)

    print("\n"+"="*60)
    print("  TAHAP 2 — Bangun & Latih Model TDNN")
    print("="*60)
    print(f"  Input dim  : {actual_dim}")
    for i,(h,ctx) in enumerate(zip(HIDDEN_DIMS, TDNN_CONTEXTS)):
        print(f"  Layer {i+1}     : {h} unit  context {ctx}")
    print(f"  Output dim : {len(word2id)}  (vocab kata)")
    print(f"  Dropout    : {DROPOUT_RATE} | LR: {LEARNING_RATE} | GradClip: {GRAD_CLIP}")

    model = TDNNModel(
        input_dim=actual_dim, hidden_dims=HIDDEN_DIMS,
        contexts=TDNN_CONTEXTS, output_dim=len(word2id),
        dropout=DROPOUT_RATE, lr=LEARNING_RATE,
    )

    history = train(model, train_ids, val_ids, sequences, labels)
    pd.DataFrame(history).to_csv(OUT_HISTORY_CSV, index=False)
    print(f"  {OUT_HISTORY_CSV} tersimpan")

    if os.path.exists(MODEL_PATH):
        model.load(MODEL_PATH)

    print("\n"+"="*60)
    print("  TAHAP 3 — Evaluasi WER + Akurasi (Val + Test)")
    print("="*60)
    eval_ids = val_ids + test_ids
    wr, ar, summary = evaluate(model, eval_ids, sequences, labels, id2word, word_refs)

    df_w = pd.DataFrame(wr)
    df_w[["audio_id","WER","S","D","I","N"]].to_csv(OUT_WER_CSV, index=False)
    df_w[["audio_id","referensi","hipotesis","WER"]].to_csv(OUT_DEC_CSV, index=False)
    pd.DataFrame(ar).to_csv(OUT_ACC_CSV, index=False)
    print(f"  {OUT_WER_CSV}\n  {OUT_DEC_CSV}\n  {OUT_ACC_CSV}")

    print("\n"+"="*60)
    print("  TAHAP 4 — Perbandingan GMM vs TDNN")
    print("="*60)
    print_comparison(summary)

    return model, summary, history


if __name__ == "__main__":
    model, summary, history = main()

  TAHAP 1 — Memuat Data & Ekstraksi MFCC dari WAV
  Folder audio  : D:\doc\2026\FINAL_CODE\data\sampled_small
  CSV transkrip : D:/doc/2026/FINAL_CODE/src/codeStat/preModelling/samplingStratified/out_samplingStratified/dataSampledSmall.csv
  Output TDNN   : D:/doc/2026/FINAL_CODE/src/codeStat/modelling/voiceModelling/srcVoiceModelling/TDNN/out/test_2
  Input dim     : 39 (MFCC 13 + delta + delta2)
  Membaca D:/doc/2026/FINAL_CODE/src/codeStat/preModelling/samplingStratified/out_samplingStratified/dataSampledSmall.csv ...
  Shape CSV : (1749, 4)
  Kolom     : ['folder', 'subfolder', 'sentences', 'Path_Data']
  Kalimat unik (kelas): 1725
  Mengekstrak MFCC dari 1749 file .wav ...
    [100/1749] diproses, 0 dilewati
    [200/1749] diproses, 0 dilewati
    [300/1749] diproses, 0 dilewati
    [400/1749] diproses, 0 dilewati
    [500/1749] diproses, 0 dilewati
    [600/1749] diproses, 0 dilewati
    [700/1749] diproses, 0 dilewati
    [800/1749] diproses, 0 dilewati
    [900/1749] diproses, 

KeyboardInterrupt: 